# Шардирование инвертированного индекса: Leiden vs Hash vs репликация термов

Каждому терму назначается шард (или несколько), документ кладётся во **все** шарды
своих термов, а запрос **накрывается** минимальным набором шардов (жадное взвешенное
покрытие его термов).

| Стратегия | Назначение терма |
|-----------|------------------|
| **Hash** | `hash(term) % N` — случайный шард |
| **Leiden** | кластер NPMI-графа со-встречаемости запросов; вне графа — hash-fallback `[N, 2N)` |
| **Leiden ×k** | кластер Лейдена **плюс** (k−1) ближайших по affinity кластеров |

**Цель исследования:** реплицируя терм в top-k кластеров, накрыть запрос одним-двумя
шардами (cover fanout → 1) ценой роста дупликации документов. Ключевой trade-off:
дупликация ↔ cover fanout ↔ recall при малом бюджете шардов.

Методика честности:
- граф строится по **train**-запросам, все метрики — на **holdout**;
- recall меряется по двум эталонам: кандидаты BM25 полного индекса и **qrels**
  (выбранные асессорами пассажи);
- при полном покрытии термов запроса recall ≡ 1 *по построению* (документ
  реплицируется во все шарды своих термов) — поэтому головная метрика не recall
  при полной маршрутизации, а **длина покрытия** и recall при бюджете меньше неё.

> Воспроизводимый прогон — `dvc repro` (стадии в `dvc.yaml`, параметры в `params.yaml`).
> Ноутбук — интерактивное зеркало на тех же функциях; пути индексов общие с пайплайном.

In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

from sharded_index import (
    PartitionConfig,
    ReplicatedTermPartition,
    TermPartition,
    assign_docs_to_shards,
    build_whoosh_indices,
    extract_query_doc_pairs,
    index_size_mb,
    tokenize,
)
from sharded_index.clusters import build_term_graph, cluster_summary_table, top_terms_per_cluster
from sharded_index.config import FIGURES_DIR, INDICES_DIR
from sharded_index.evaluation import (
    build_ground_truth,
    compute_routing_recall,
    cover_fanouts,
    recall_by_fanout,
)
from sharded_index.plots import (
    plot_cluster_size_distribution,
    plot_cluster_wordclouds,
    plot_fanout_distribution,
    plot_graph_clusters,
    plot_inter_cluster_heatmap,
    plot_recall_vs_fanout,
    plot_replication_tradeoff,
    plot_tsne_projection,
)

plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12})

## Конфигурация

Значения по умолчанию совпадают с `params.yaml`; пути берутся из
`sharded_index.config` (привязаны к корню проекта), поэтому ноутбук работает
из каталога `notebooks/` и переиспользует артефакты `dvc repro`.

In [ ]:
TRAIN_RATIO = 0.8          # доля запросов на построение графа; хвост — holdout
N_REPLICAS = 2             # k для стратегии Leiden ×k
N_EVAL = 100_000           # holdout-запросов в оценке recall
N_FANOUT_SAMPLE = 100_000  # holdout-запросов для распределения cover fanout
MAX_FANOUT_SWEEP = 5       # бюджет шардов в развёртке recall-vs-fanout

HASH_ROOT = INDICES_DIR / "hash"
LEIDEN_ROOT = INDICES_DIR / "leiden"
REPL_ROOT = INDICES_DIR / f"leiden_r{N_REPLICAS}"
FULL_ROOT = INDICES_DIR / "full"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Данные и сплит

MS MARCO v2.1: пассажи, отмеченные асессорами (до 3 на запрос); тексты нормализуются,
`doc_id` — MD5 текста (дедупликация). Запросы делятся на train (граф) и holdout
(оценка); qrels запроса — его собственные выбранные пассажи.

In [ ]:
dataset = load_dataset("microsoft/ms_marco", "v2.1")
pairs = extract_query_doc_pairs(
    dataset["train"], max_rows=200_000, max_passages_per_query=3, selected_only=True
)

docs = {p["doc_id"]: p["doc_text"] for p in pairs}
queries = list(dict.fromkeys(p["query"] for p in pairs))  # уникальные, детерминированно

n_train = int(len(queries) * TRAIN_RATIO)
train_queries, holdout_queries = queries[:n_train], queries[n_train:]

qrels: dict[str, set[str]] = defaultdict(set)  # запрос -> его релевантные документы
for p in pairs:
    qrels[p["query"]].add(p["doc_id"])
qrels = dict(qrels)

print(f"Пар: {len(pairs):,}; документов: {len(docs):,}")
print(f"Запросов: {len(queries):,} = {len(train_queries):,} train + {len(holdout_queries):,} holdout")

## 2. Разбиения термов

- `leiden_core` — Leiden-кластеры NPMI-графа, построенного **только по train-запросам**;
- `hash` — случайный бейзлайн (столько же шардов, `node_strength` заимствован для
  сопоставимого ранжирования);
- `leiden` — ядро + hash-fallback `[N, 2N)` для термов корпуса вне графа;
- `leiden ×k` — графовые термы реплицируются в top-k кластеров по NPMI-affinity
  (fallback-термы не реплицируются).

In [ ]:
train_set = set(train_queries)
graph_texts = [p["query"] for p in pairs if p["query"] in train_set]

leiden_core = TermPartition.from_corpus(
    graph_texts,
    config=PartitionConfig(
        min_df=2, max_df_ratio=0.5, min_pair_count=2, min_npmi=0.0,
        leiden_resolution=1.0, leiden_seed=42,
    ),
)
print(f"Кластеров Leiden: {leiden_core.n_shards}, термов в графе: {leiden_core.n_terms:,}")

corpus_terms = set()
for text in docs.values():
    corpus_terms.update(tokenize(text))
print(f"Термов в корпусе: {len(corpus_terms):,} "
      f"(fallback для {len(corpus_terms) - leiden_core.n_terms:,})")

hash_partition = TermPartition.from_hashing(
    corpus_terms, n_shards=leiden_core.n_shards, node_strength=leiden_core.node_strength
)
leiden_partition = leiden_core.with_hash_fallback(corpus_terms)
replicated = ReplicatedTermPartition.from_partition(leiden_partition, N_REPLICAS)

strategies = {
    "Hash": (hash_partition, HASH_ROOT),
    "Leiden": (leiden_partition, LEIDEN_ROOT),
    f"Leiden ×{N_REPLICAS}": (replicated, REPL_ROOT),
}
print(f"\nHash: {hash_partition.n_shards} шардов | Leiden: {leiden_partition.n_shards} шардов")
print(f"Leiden ×{N_REPLICAS}: replication factor {replicated.replication_factor:.2f}")

## 3. Документы по шардам и Whoosh-индексы

Дупликация = среднее число шардов на документ; у реплицированной стратегии она
заведомо выше — это цена, которую мы платим за короткое покрытие запроса.

In [ ]:
doc_shards: dict[str, dict[str, set[int]]] = {}
duplication: dict[str, float] = {}

for name, (part, root) in strategies.items():
    d2s, docs_by_shard = assign_docs_to_shards(docs, part)
    doc_shards[name] = d2s
    duplication[name] = float(np.mean([len(s) for s in d2s.values()]))
    print(f"{name}: {len(docs_by_shard)} шардов, дупликация {duplication[name]:.2f}x")
    build_whoosh_indices(docs_by_shard, root)

build_whoosh_indices({0: docs}, FULL_ROOT)  # единый индекс — эталон

full_mb = index_size_mb(FULL_ROOT)
size_mb = {name: index_size_mb(root) for name, (_, root) in strategies.items()}
print("\n" + " | ".join(
    [f"Full: {full_mb:.1f} MB"]
    + [f"{n}: {s:.1f} MB ({s / full_mb:.1f}x)" for n, s in size_mb.items()]
))

## 4. Cover fanout

Сколько шардов нужно, чтобы жадно накрыть все термы запроса. Для одиночного
назначения это обычный fanout (по шарду на кластер термов); реплики существуют,
чтобы покрытие стало короче.

In [ ]:
sample = holdout_queries[:N_FANOUT_SAMPLE]
cover = {name: cover_fanouts(sample, part) for name, (part, _) in strategies.items()}

for name, values in cover.items():
    print(f"{name:12s} mean={np.mean(values):.2f}, median={np.median(values):.0f}, "
          f"p95={np.percentile(values, 95):.0f}")

plot_fanout_distribution(cover, save_path=FIGURES_DIR / "fanout_distribution.pdf")

## 5. Recall на holdout

Два эталона: кандидаты BM25 полного индекса (sanity: при полной маршрутизации
recall обязан быть 100%) и qrels. Qrels-recall при полной маршрутизации — почти
лексический потолок достижимости (релевантный пассаж без общих термов с запросом
недостижим в принципе).

In [ ]:
eval_queries = holdout_queries[:N_EVAL]
ground_truth = build_ground_truth(eval_queries, FULL_ROOT / "shard_000")
eval_qrels = {q: qrels[q] for q in eval_queries}

for name, (part, _) in strategies.items():
    sanity = compute_routing_recall(eval_queries, ground_truth, part, doc_shards[name])
    by_qrels = compute_routing_recall(eval_queries, eval_qrels, part, doc_shards[name])
    print(f"{name:12s} sanity={sanity.recall:.1%} (инвариант, должно быть 100%) | "
          f"qrels={by_qrels.recall:.1%}")

## 6. Recall при бюджете шардов

Опрашиваются только первые k шардов жадного покрытия. Здесь и живёт исследуемый
эффект: чем короче покрытие, тем больший recall достаётся первому шарду.

In [ ]:
recall_curves = {
    name: recall_by_fanout(eval_queries, ground_truth, part, doc_shards[name],
                           max_fanout=MAX_FANOUT_SWEEP)
    for name, (part, _) in strategies.items()
}
qrels_curves = {
    name: recall_by_fanout(eval_queries, eval_qrels, part, doc_shards[name],
                           max_fanout=MAX_FANOUT_SWEEP)
    for name, (part, _) in strategies.items()
}
mean_cover = {name: float(np.mean(values)) for name, values in cover.items()}

plot_recall_vs_fanout(recall_curves, mean_cover,
                      save_path=FIGURES_DIR / "recall_vs_fanout.pdf")

## 7. Trade-off репликации и сводка

Главный график исследования: сколько дупликации покупает какое сокращение
покрытия (и какой recall при бюджете в один шард).

In [ ]:
tradeoff = {
    name: {
        "duplication": duplication[name],
        "cover_fanout_mean": mean_cover[name],
        "recall_at_1": recall_curves[name][1],
    }
    for name in strategies
}
plot_replication_tradeoff(tradeoff, save_path=FIGURES_DIR / "duplication_vs_fanout.pdf")

In [ ]:
summary = pd.DataFrame([
    {
        "Strategy": name,
        "Shards": part.n_shards,
        "Duplication": f"{duplication[name]:.1f}x",
        "Storage": f"{size_mb[name] / full_mb:.1f}x",
        "Cover fanout": f"{mean_cover[name]:.2f} (p95={np.percentile(cover[name], 95):.0f})",
        "Recall@1": f"{recall_curves[name][1]:.1%}",
        "Qrels@1": f"{qrels_curves[name][1]:.1%}",
        "Qrels full": f"{compute_routing_recall(eval_queries, eval_qrels, part, doc_shards[name]).recall:.1%}",
    }
    for name, (part, _) in strategies.items()
])
summary

## 8. Анализ кластеров

Только семантическое ядро (`leiden_core`) — термы NPMI-графа и их Leiden-кластеры.

In [ ]:
plot_cluster_size_distribution(leiden_core.clusters_df,
                               save_path=FIGURES_DIR / "cluster_size_distribution.pdf")

sizes = leiden_core.clusters_df["size"]
print(f"Кластеров: {len(sizes)}, медиана размера: {sizes.median():.0f}, "
      f"среднее: {sizes.mean():.1f}, min/max: {sizes.min()}/{sizes.max()}")

In [ ]:
top_terms_per_cluster(leiden_core, n_clusters=10, n_terms=15)

In [ ]:
plot_cluster_wordclouds(leiden_core, top_n=25,
                        save_path=FIGURES_DIR / "word_clouds_top25.pdf")

In [ ]:
G = build_term_graph(leiden_core)
print(f"Граф: |V|={G.number_of_nodes():,}, |E|={G.number_of_edges():,}")

plot_graph_clusters(G, leiden_core, max_nodes=1000, seed=42,
                    save_path=FIGURES_DIR / "npmi_graph_clusters.pdf")

In [ ]:
plot_tsne_projection(G, leiden_core, top_n_clusters=25, seed=42,
                     save_path=FIGURES_DIR / "tsne_2d_projection.pdf")

In [ ]:
cluster_summary_table(leiden_core, top_n=25).style.format({
    "total_strength": "{:.1f}",
    "avg_strength": "{:.3f}",
    "intra_weight": "{:.1f}",
    "density": "{:.4f}",
}).background_gradient(subset=["size", "total_strength", "density"], cmap="Blues")

In [ ]:
plot_inter_cluster_heatmap(leiden_core, top_n_clusters=25,
                           save_path=FIGURES_DIR / "inter_cluster_heatmap.pdf")